# 02. 第一個 LangGraph 應用

本 notebook 將帶您建立第一個完整的 LangGraph 對話應用。

---

## 🎯 學習目標

完成本章節後，您將能夠：
- ✅ 理解 Graph 的完整建構流程
- ✅ 使用 `add_messages` reducer 管理對話
- ✅ 實作簡單的聊天機器人節點
- ✅ 使用串流 (stream) 觀察執行過程

---

## 📊 本章架構圖

```
┌─────────────────────────────────────────────────────────┐
│                   簡單聊天機器人架構                      │
├─────────────────────────────────────────────────────────┤
│                                                         │
│   ┌─────────────┐                   ┌─────────────┐    │
│   │   START     │                   │     END     │    │
│   └──────┬──────┘                   └──────▲──────┘    │
│          │                                 │           │
│          ▼                                 │           │
│   ┌─────────────────────────────────────────┐          │
│   │              chatbot                    │          │
│   │         (處理訊息並回覆)                 │──────────┘
│   └─────────────────────────────────────────┘          │
│                                                         │
└─────────────────────────────────────────────────────────┘
```

In [1]:
from typing import TypedDict, Annotated
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages

---

## 2.1 定義狀態結構

### 什麼是 Reducer？

當多個節點更新同一個狀態欄位時，Reducer 決定如何合併這些更新：

```
沒有 Reducer:  新值 覆蓋 舊值
有 Reducer:    新值 = reducer(舊值, 更新值)
```

### add_messages Reducer

```python
# 沒有 reducer - 會覆蓋
messages: list  
# ["Hello"] + ["World"] → ["World"]  ❌

# 有 reducer - 會累加
messages: Annotated[list, add_messages]
# ["Hello"] + ["World"] → ["Hello", "World"]  ✅
```

In [2]:
class ChatState(TypedDict):
    """對話狀態
    
    Attributes:
        messages: 對話歷史，使用 add_messages reducer 自動累加
    """
    messages: Annotated[list, add_messages]

print("✅ 狀態定義完成")
print("\n💡 add_messages 會自動將新訊息附加到列表末尾")

✅ 狀態定義完成

💡 add_messages 會自動將新訊息附加到列表末尾


---

## 2.2 定義節點函數

節點函數的規則：

| 規則 | 說明 |
|------|------|
| 輸入 | 接收完整的 State |
| 輸出 | 返回要更新的欄位（dict） |
| 不變性 | 不要直接修改 state，返回新值 |

In [3]:
def echo_chatbot(state: ChatState) -> dict:
    """模擬聊天機器人 - 回傳處理後的訊息
    
    這是一個簡單的 echo bot，實際應用中會換成 LLM
    
    Args:
        state: 包含 messages 的狀態
        
    Returns:
        包含新回覆訊息的 dict
    """
    # 取得最後一條訊息
    last_message = state["messages"][-1] if state["messages"] else None
    
    # 處理訊息內容
    if last_message:
        if hasattr(last_message, 'content'):
            content = last_message.content
        elif isinstance(last_message, dict):
            content = last_message.get('content', str(last_message))
        else:
            content = str(last_message)
    else:
        content = "No message"
    
    # 生成回覆
    response = f"🤖 收到您的訊息: '{content}'"
    print(f"  📨 處理訊息: {content[:30]}..." if len(content) > 30 else f"  📨 處理訊息: {content}")
    
    return {"messages": [{"role": "assistant", "content": response}]}

print("✅ 節點函數定義完成")

✅ 節點函數定義完成


---

## 2.3 建構圖

建構圖的三個步驟：

```python
# 1. 建立圖實例
graph = StateGraph(State)

# 2. 添加節點
graph.add_node("name", function)

# 3. 添加邊
graph.add_edge(START, "name")
graph.add_edge("name", END)
```

In [4]:
# 建立 StateGraph
graph = StateGraph(ChatState)

# 新增節點
graph.add_node("chatbot", echo_chatbot)

# 定義流程：START → chatbot → END
graph.add_edge(START, "chatbot")
graph.add_edge("chatbot", END)

# 編譯成可執行的應用
app = graph.compile()

print("✅ Graph 建構完成！")
print("\n📊 流程: START → chatbot → END")

✅ Graph 建構完成！

📊 流程: START → chatbot → END


---

## 2.4 視覺化

In [5]:
print("📊 圖結構 (Mermaid):")
print("=" * 40)
print(app.get_graph().draw_mermaid())

📊 圖結構 (Mermaid):
---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	chatbot(chatbot)
	__end__([<p>__end__</p>]):::last
	__start__ --> chatbot;
	chatbot --> __end__;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc



---

## 2.5 執行對話

使用 `invoke()` 方法執行圖：

In [6]:
print("🚀 執行對話：")
print("=" * 40)

# 單次對話
result = app.invoke({
    "messages": [{"role": "user", "content": "你好！請介紹一下你自己"}]
})

print("=" * 40)
print("\n📋 結果：")
print(f"  訊息數量: {len(result['messages'])}")
print(f"  最後回覆: {result['messages'][-1]}")

🚀 執行對話：
  📨 處理訊息: 你好！請介紹一下你自己

📋 結果：
  訊息數量: 2
  最後回覆: content="🤖 收到您的訊息: '你好！請介紹一下你自己'" additional_kwargs={} response_metadata={} id='f3e217e9-e926-496b-bb68-ab2f68a0bf28'


---

## 2.6 串流輸出

使用 `stream()` 可以逐步觀察每個節點的執行：

```python
for chunk in app.stream(input):
    # chunk = {"node_name": {...更新的狀態...}}
```

In [7]:
print("🔄 串流輸出：")
print("=" * 40)

for chunk in app.stream({"messages": [{"role": "user", "content": "Hello world!"}]}):
    for node_name, values in chunk.items():
        print(f"\n📍 節點: {node_name}")
        print(f"   更新: {list(values.keys())}")
        if 'messages' in values:
            print(f"   訊息: {values['messages'][-1]}")

print("\n" + "=" * 40)
print("✅ 串流完成")

🔄 串流輸出：
  📨 處理訊息: Hello world!

📍 節點: chatbot
   更新: ['messages']
   訊息: {'role': 'assistant', 'content': "🤖 收到您的訊息: 'Hello world!'"}

✅ 串流完成


---

## 2.7 多輪對話模擬

由於我們使用了 `add_messages` reducer，可以模擬多輪對話：

In [8]:
print("💬 多輪對話模擬：")
print("=" * 40)

# 模擬多輪對話
conversation = [
    "你好！",
    "今天天氣如何？",
    "謝謝你的回覆！"
]

messages = []
for i, user_msg in enumerate(conversation, 1):
    print(f"\n--- 第 {i} 輪 ---")
    messages.append({"role": "user", "content": user_msg})
    print(f"👤 用戶: {user_msg}")
    
    result = app.invoke({"messages": messages})
    assistant_msg = result["messages"][-1]
    print(f"🤖 助手: {assistant_msg.get('content', assistant_msg) if isinstance(assistant_msg, dict) else assistant_msg}")
    
    messages = result["messages"]

print("\n" + "=" * 40)
print(f"📊 總共 {len(messages)} 條訊息")

💬 多輪對話模擬：

--- 第 1 輪 ---
👤 用戶: 你好！
  📨 處理訊息: 你好！
🤖 助手: content="🤖 收到您的訊息: '你好！'" additional_kwargs={} response_metadata={} id='11e7c754-a8ac-4112-b5df-b8dcd7d56490'

--- 第 2 輪 ---
👤 用戶: 今天天氣如何？
  📨 處理訊息: 今天天氣如何？
🤖 助手: content="🤖 收到您的訊息: '今天天氣如何？'" additional_kwargs={} response_metadata={} id='ff9f8f6f-f174-4dd8-bb77-7533a3c02ddb'

--- 第 3 輪 ---
👤 用戶: 謝謝你的回覆！
  📨 處理訊息: 謝謝你的回覆！
🤖 助手: content="🤖 收到您的訊息: '謝謝你的回覆！'" additional_kwargs={} response_metadata={} id='c7babfd5-f236-42ec-94d2-2800a393f203'

📊 總共 6 條訊息


---

## 💡 重點回顧

### invoke vs stream

| 方法 | 用途 | 返回值 |
|------|------|--------|
| `invoke(state)` | 執行並等待完成 | 最終 State |
| `stream(state)` | 逐步執行 | 每個節點的更新 |

### add_messages 的作用

```python
# 節點返回
return {"messages": [new_msg]}

# 實際效果
state["messages"] = old_messages + [new_msg]
```

### 節點函數模板

```python
def my_node(state: State) -> dict:
    # 1. 讀取狀態
    data = state["key"]
    
    # 2. 處理邏輯
    result = process(data)
    
    # 3. 返回更新
    return {"key": result}
```

---

## 📝 練習題

1. **修改回覆**：讓 chatbot 回覆時加上時間戳記
2. **新增節點**：加入一個「翻譯」節點，將回覆轉成大寫
3. **統計功能**：在 State 中加入 `message_count` 欄位追蹤訊息數
4. **進階**：嘗試串接多個處理節點

---

下一步：[03. 狀態管理](03_state_management.ipynb)